# 03 지표 계산과 검색 개수

목표: Hit·Recall·Mean Reciprocal Rank·normalized Discounted Cumulative Gain 손계산과 코드 대조.

**준비:** 저장소 터미널에서 python -m pip install -r requirements/week11-offline.txt 실행.
VS Code 커널에서 저장소 .venv의 **Python 3.11** 선택 후 위에서 아래로 실행합니다.
각 파일은 다른 Notebook의 변수를 요구하지 않습니다. GPU·Ollama·API 키는 필요 없습니다.

**데이터:** 교수 작성 가상 쇼핑 정책입니다. 실제 기업 규정이나 고객 기록이 아닙니다.
핵심 단계는 수업에서 함께 실행하고 확장 실험은 복습용입니다. [실습 안내](../README.md).


In [ ]:
from pathlib import Path
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week11/lab/evaluation.py").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError("저장소 안에서 Notebook을 실행하세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("저장소 .venv의 Python 3.11 커널을 선택하세요.")
import pandas as pd
from IPython.display import display
from week11.lab.evaluation import load_data, Config, evaluate, summarize, ranking_metrics

docs, questions = load_data()
dev = [q for q in questions if q["split"] == "dev"]
test = [q for q in questions if q["split"] == "test"]
print("환경 준비 완료", sys.version.split()[0], "문서", len(docs), "질문", len(questions))

## 1. 복수 정답의 Hit와 Recall

정답 두 개 중 하나 검색이면 Hit=1, Recall=0.5입니다.

In [ ]:
display(
    pd.DataFrame(
        [
            ranking_metrics(["D03", "D08", "D09"], ["D03", "D04"], 3),
            ranking_metrics(["D08", "D03", "D04"], ["D03", "D04"], 3),
        ],
        index=["하나만", "둘 다"],
    )
)

**확인과 해석**

하나라도 찾았는지와 필요한 근거를 모두 찾았는지는 다릅니다.

## 2. 순위 할인

MRR은 첫 정답 순위, nDCG는 여러 관련 문서의 순위를 반영합니다. 이 실습은 이진 관련도입니다.

In [ ]:
import math

metrics = ranking_metrics(["X", "A", "B"], ["A", "B"], 3)
manual = (1 / math.log2(3) + 1 / math.log2(4)) / (1 + 1 / math.log2(3))
print(metrics, manual)
assert abs(metrics["ndcg"] - manual) < 1e-12

**확인과 해석**

이상 순위 분모는 min(k, 정답 문서 수)까지 계산합니다.

## 3. 라이브러리 교차 검증

정의와 ties 처리를 맞춰 비교합니다.

In [ ]:
from sklearn.metrics import ndcg_score

expected = ndcg_score([[0, 1, 1]], [[3, 2, 1]], k=3)
assert abs(expected - metrics["ndcg"]) < 1e-12
print(expected)

**확인과 해석**

직접 구현한 지표도 테스트 대상입니다.

## 4. 답변 불가 분모

정답 문서가 비어 있으면 Recall은 정의되지 않습니다.

In [ ]:
print("답변 불가:", ranking_metrics(["A"], [], 3))
print("검색 실패:", ranking_metrics([], ["A"], 3))

**확인과 해석**

None과 검색 실패 0점을 구분합니다.

## 5. 검색 개수 k

Recall 상승이 생성 품질 상승을 보장하지 않습니다.

In [ ]:
rows = [
    {"k": k, **summarize(evaluate(docs, dev, Config(f"k{k}", "char", True, k)))}
    for k in [1, 2, 3, 5, 10]
]
display(pd.DataFrame(rows)[["k", "hit", "recall", "mrr", "ndcg", "false_accept"]])

**확인과 해석**

LLM 생성이 없으므로 답변 품질·토큰 비용을 측정했다고 말할 수 없습니다.

## 6. 중복 차단

같은 정답 반복으로 Recall이 부풀려지는 오류를 막습니다.

In [ ]:
try:
    ranking_metrics(["A", "A"], ["A"], 3)
except ValueError as error:
    print("중복 차단:", error)

**확인과 해석**

실제 청크 검색은 청크/문서 중 평가 단위를 먼저 고정합니다.

## 확장 실험과 프로젝트 적용

1. k=1,3,5를 같은 질문에서 비교합니다.
2. 서비스의 성공이 전체 근거 확보인지 정의합니다.
3. 생성 모델 연결 시 context 길이·정확성·인용·지연을 측정합니다.

**실행 실패 시:** ModuleNotFoundError는 커널과 설치 터미널이 같은 .venv인지 확인합니다.
ROOT 오류는 저장소 안에서 파일을 열면 해결됩니다. 변수가 섞였다면 Restart Kernel 후 Run All.
예외 실험은 try/except로 설명하므로 정상 실행에서는 빨간 오류로 끝나지 않습니다.

**기록:** 변경 설정 / 예상 / 실제 관찰 / 실패 질문 ID / 다음 행동을 5줄로 적습니다.
개인 프로젝트 원문·연락처·API 키는 공개 GitHub에 올리지 않습니다.
